In [1]:
import gymnasium as gym
import highway_env

import jax
import jax.numpy as jnp
from flax import nnx
from functools import partial

<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


Below is the policy network $\pi_\theta : \mathcal{S} \to \Delta(\mathcal{A})$,<br>
with $\mathcal{S} = \mathbb{R}^{5 \times 5}$ and $\mathcal{A} = \lbrace 0, 1, 2, 3, 4 \rbrace$.

Given a state $s \in \mathcal{S}$, it outputs a probability distribution $\pi_\theta(\cdot \mid s)$ over $\mathcal{A}$.

In [2]:
class MLP(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.l1 = nnx.Linear(25, 64, rngs=rngs)
        self.l2 = nnx.Linear(64, 64, rngs=rngs)
        self.out = nnx.Linear(64, 5, rngs=rngs)

    def __call__(self, x):
        x = x.reshape(*x.shape[:-2], -1)
        x = nnx.relu(self.l1(x))
        x = nnx.relu(self.l2(x))
        return self.out(x)

@partial(jax.jit, static_argnums=0)
def apply(graphdef, params, x):
    return nnx.merge(graphdef, params)(x)

In [3]:
gym.register_envs(highway_env)

graphdef, params = nnx.split(MLP(nnx.Rngs(0)))

env = gym.make("highway-v0", render_mode="human")
obs, info = env.reset()

print("Action space     :", env.action_space)
print("Observation space:", env.observation_space)

stop = False
while not stop:
    q_values = apply(graphdef, params, jnp.asarray(obs))
    action = int(jnp.argmax(q_values))

    obs, reward, terminated, truncated, info = env.step(action)

#-------------------------import simplemlp as smlp---------------------------------------------------------------------------

    stop = terminated or truncated
env.close()

Action space     : Discrete(5)
Observation space: Box(-inf, inf, (5, 5), float32)
